<!-- beginner-banner-v2 -->

> 🧭 <strong>비개발자 수강생 안내</strong> — 이 노트북에서 새로 배우는 것: 텍스트를 <strong>숫자 좌표(임베딩)</strong> 로 바꾸고, ChromaDB 로 영속화.
>
> - 📖 강의 페이지: <a href="https://siapapa.github.io/day1/06-embedding-chromadb/" target="_blank" rel="noopener noreferrer">day1/06-embedding-chromadb</a>
> - 🆕 처음이라면 → <a href="https://siapapa.github.io/beginners-guide/" target="_blank" rel="noopener noreferrer">비개발자 학습 가이드</a>
> - 🔤 모르는 단어 → <a href="https://siapapa.github.io/appendix/glossary/" target="_blank" rel="noopener noreferrer">용어 사전</a>
> - 🛠️ 환경/접속 막힘 → <a href="https://siapapa.github.io/setup/" target="_blank" rel="noopener noreferrer">사전 준비</a> · <a href="https://siapapa.github.io/appendix/troubleshooting/" target="_blank" rel="noopener noreferrer">트러블슈팅</a>
>
> 외부 링크는 새 탭으로 열리도록 설정돼 있어 Colab 의 리디렉션 경고 페이지를 거치지 않습니다.<br/>
> <strong>셀은 위에서 아래로 차례대로 실행</strong>하세요. 시연용 코드(<code>구경만 하세요</code> 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 05. 임베딩 + ChromaDB 영속화
> Day 1 · 6H · 소요 약 50분

## 학습 목표

- 임베딩 벡터와 코사인 유사도 개념을 수치적으로 체감한다.
- ChromaDB 에 문서를 영속화하고 Top-K 검색을 수행한다.
- LlamaIndex + ChromaDB 통합으로 영속 인덱스를 만든다.

> 이 노트북의 `./chroma_db` 디렉토리는 **본 노트북 전용**입니다. 이후 노트북(특히 13번 LCEL RAG)은 langchain-chroma를 독립적으로 사용합니다.

In [ ]:
%pip install -q chromadb llama-index llama-index-vector-stores-chroma \
    llama-index-embeddings-openai llama-index-llms-openai openai \
    numpy matplotlib

In [ ]:
# k-font-setup-v1
# ============================================================
# 🇰🇷 matplotlib 한글 폰트 자동 설정
# ------------------------------------------------------------
# 차트의 한글 라벨이 □ 박스로 깨지지 않도록 Nanum 폰트를 설치하고
# matplotlib 에 등록한다. 셀 재실행 시에도 안전(이미 설치돼 있으면 즉시
# 통과). 이 노트북에서 matplotlib 를 import 하지 않더라도 무해하다.
# ============================================================
import os, subprocess, warnings

# 1) Nanum 폰트 설치 (Colab apt 는 sudo 불필요)
try:
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   check=False, capture_output=True, timeout=120)
    subprocess.run(["fc-cache", "-fv"],
                   check=False, capture_output=True, timeout=60)
except Exception:
    pass

# 2) matplotlib 폰트 매니저에 NanumGothic 등록 → 즉시 활성화
try:
    import matplotlib as _mpl
    import matplotlib.pyplot as _plt
    from matplotlib import font_manager as _fm

    nanum_paths = [p for p in _fm.findSystemFonts(fontpaths=None)
                   if "Nanum" in p or "nanum" in p]
    for p in nanum_paths:
        try:
            _fm.fontManager.addfont(p)
        except Exception:
            pass

    if nanum_paths:
        _plt.rcParams["font.family"] = "NanumGothic"
        _plt.rcParams["axes.unicode_minus"] = False  # 마이너스 부호 깨짐 방지
        print("✅ matplotlib 한글 폰트 활성화: NanumGothic ({} 개)".format(len(nanum_paths)))
    else:
        # 비-Colab 폴백: 시스템에 깔린 한글 폰트 사용
        fallback = None
        for fname in ["Malgun Gothic", "AppleGothic", "Noto Sans CJK KR", "Noto Sans KR"]:
            if any(fname.replace(" ", "").lower() in p.lower()
                   for p in _fm.findSystemFonts()):
                fallback = fname
                break
        if fallback:
            _plt.rcParams["font.family"] = fallback
            _plt.rcParams["axes.unicode_minus"] = False
            print("✅ matplotlib 한글 폰트 폴백:", fallback)
        else:
            warnings.warn("⚠️ 한글 폰트를 찾지 못했습니다 — 차트의 한글이 □ 로 보일 수 있습니다.")
except ImportError:
    pass  # matplotlib 미설치 — plot 없는 환경에서는 무시


In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# 임베딩 호출에 OpenAI 키가 필수.
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")

## 임베딩 = 텍스트를 숫자 벡터로

```
"환자가 두통을 호소합니다"  →  [0.12, -0.34, 0.56, ...]  (1,536차원)
"머리가 아파요"           →  [0.11, -0.32, 0.55, ...]  (1,536차원)
"오늘 날씨가 좋습니다"      →  [0.78,  0.45, -0.12, ...] (1,536차원)
```

**코사인 유사도**: 두 벡터가 이루는 각도의 코사인. `1.0`이면 매우 유사, `0.0`이면 무관.

$$\cos(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

In [ ]:
# 임베딩 = 텍스트를 의미를 보존한 "고차원 좌표(벡터)" 로 변환하는 작업.
# 두 텍스트의 코사인 유사도가 1에 가까우면 의미가 비슷, 0이면 무관, -1이면 정반대 의미.
from openai import OpenAI
import numpy as np

# OpenAI 공식 SDK 클라이언트. 환경변수 OPENAI_API_KEY 가 자동으로 사용됩니다.
client = OpenAI()

def get_embedding(text: str) -> list[float]:
    """문장 → 1,536 차원 실수 벡터(list[float]) 반환."""
    resp = client.embeddings.create(input=text, model="text-embedding-3-small")
    # resp.data 는 입력 1개당 1개의 결과를 담은 리스트. 첫 번째 결과의 .embedding 이 실제 벡터.
    return resp.data[0].embedding

def cosine_similarity(a, b) -> float:
    """두 벡터의 코사인 유사도. 공식: dot(a,b) / (|a| * |b|)."""
    a, b = np.array(a), np.array(b)
    # np.dot(a, b)         : 두 벡터의 내적 (대응 원소를 곱해서 모두 더함)
    # np.linalg.norm(a)    : 벡터의 길이 (sqrt(원소 제곱의 합))
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 의미가 비슷한 의료 표현 2개 + 다른 주제(날씨) 3개를 섞어 거리 차이를 시각적으로 본다.
sentences = [
    "환자가 심한 두통을 호소합니다",
    "머리가 깨질 듯이 아파요",
    "복부에 통증이 있습니다",
    "오늘 서울 날씨가 맑습니다",
    "내일 비가 올 예정입니다",
]
# 리스트 컴프리헨션: `[표현 for 변수 in 반복가능객체]` — 각 문장에 대해 한 번씩 임베딩을 받는다.
embeddings = [get_embedding(s) for s in sentences]
print(f"Embedding dim: {len(embeddings[0])}")  # 1536 출력 예상
print(f"Sample (first 8 dims): {embeddings[0][:8]}")

In [ ]:
# 5×5 유사도 행렬을 만들고 히트맵으로 시각화 — 의료 문장끼리(0,1) 점수가 높고
# 의료 vs 날씨 사이는 점수가 낮게 나오는 것을 한눈에 본다.
import matplotlib.pyplot as plt
import matplotlib


n = len(sentences)
sim_matrix = np.zeros((n, n))   # n×n 영행렬을 만들고 한 칸씩 채워 넣음
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = cosine_similarity(embeddings[i], embeddings[j])

print("Similarity matrix:")
# np.round 로 소수 셋째 자리에서 반올림해 출력을 깔끔하게.
print(np.round(sim_matrix, 3))

# 히트맵 그리기
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)  # 색상 범위를 0~1 로 고정
plt.colorbar(im)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(range(n))
ax.set_yticklabels([f"[{i}] {s[:15]}" for i, s in enumerate(sentences)])
# 각 셀 가운데에 숫자 라벨을 그려서 색만으로 안 보일 때도 값 확인이 쉽게.
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i][j]:.2f}", ha="center", va="center", fontsize=9)
plt.title("Cosine Similarity Matrix")
plt.tight_layout()
plt.show()

## ChromaDB — 벡터 영속 저장소

`PersistentClient(path=...)` 로 디스크 경로를 지정하면 파이썬 프로세스를 재시작해도 데이터가 유지됩니다. Colab에서는 Drive 마운트로 세션 간 보존도 가능합니다.

In [ ]:
# ChromaDB — 벡터 영속 저장소(=벡터 DB). 임베딩 벡터를 디스크에 보관하고 빠르게 검색하게 해 줍니다.
import chromadb

# PersistentClient 는 지정 경로에 SQLite + 인덱스 파일을 만들어 둡니다.
# 노트북을 재시작해도 같은 path 로 다시 열면 이전 데이터가 그대로 살아 있습니다.
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Collection = "테이블" 같은 단위. 같은 이름이 이미 있으면 재사용, 없으면 새로 만든다.
# `hnsw:space=cosine` → 검색 시 거리 측정 방식을 코사인으로 지정 (기본은 L2 거리).
collection = chroma_client.get_or_create_collection(
    name="hospital_docs",
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection: {collection.name}, existing docs: {collection.count()}")

In [ ]:
# 8개 짧은 문서를 ChromaDB 에 적재. 각 문서마다 (id, 본문, 메타데이터) 3 종 세트가 필요합니다.
documents = [
    "서울중앙병원 내과에는 김철수(심장), 이영희(호흡기), 신민아(소화기) 전문의가 있습니다.",
    "외과는 박민수(일반외과), 정수진(흉부외과), 권혁준(혈관외과)이 근무합니다.",
    "진료 시간은 평일 09:00-18:00, 토요일 09:00-13:00입니다.",
    "응급실은 24시간 운영되며, 야간 당직의가 상주합니다.",
    "입원 병실은 1인실(25만원/일), 2인실(15만원/일), 4인실(8만원/일)입니다.",
    "외래 환자 주차는 3시간 무료이며, 이후 30분당 1,000원입니다.",
    "진단서 발급은 1층 제증명 창구에서 가능하며, 소요 시간은 약 30분입니다.",
    "소아과에는 최동현, 강미래, 문서영 전문의가 소아청소년 질환을 진료합니다.",
]

# section: 사람이 검색할 카테고리, doc_type: where 필터에 쓸 분류 라벨.
metadatas = [
    {"section": "내과",   "doc_type": "department"},
    {"section": "외과",   "doc_type": "department"},
    {"section": "진료시간","doc_type": "guide"},
    {"section": "응급실", "doc_type": "guide"},
    {"section": "입원",   "doc_type": "guide"},
    {"section": "주차",   "doc_type": "guide"},
    {"section": "제증명", "doc_type": "guide"},
    {"section": "소아과", "doc_type": "department"},
]

# upsert: 같은 id 가 이미 있으면 갱신, 없으면 삽입 (= update + insert).
# 셀을 여러 번 돌려도 중복이 쌓이지 않아 학습용으로 안전합니다.
# ids 는 문자열 리스트 — `f"doc_{i}"` 가 doc_0, doc_1, … 을 생성.
# Chroma 가 documents 텍스트를 자동으로 임베딩까지 처리해 줍니다 (설치 시 기본 임베딩 함수 사용).
collection.upsert(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    metadatas=metadatas,
)
print(f"Collection now has {collection.count()} docs.")

In [ ]:
# Top-K 검색: 질문과 의미가 가까운 상위 K 개 문서를 가져온다.
# 쿼리 텍스트는 자동으로 임베딩 → 벡터 비교 → 거리 기준으로 정렬됩니다.
results = collection.query(
    query_texts=["내과 의사가 누구인가요?"],
    n_results=3,   # K=3
)

# Chroma 의 결과 형식: 키별로 list[list]. 첫 번째 [0] 인덱스는 "첫 번째 쿼리에 대한 결과"를 뜻함.
# zip 으로 (문서, 메타, 거리) 를 한 번에 묶어 한 행씩 출력.
print("Query: 내과 의사가 누구인가요?\n")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    # `distances` 는 거리(작을수록 가까움). 코사인 거리는 (1 - 코사인 유사도) 이므로 1에서 빼서 점수로 변환.
    sim = 1 - dist
    print(f"[{i+1}] sim={sim:.3f} section={meta['section']}")
    print(f"    {doc}")

In [ ]:
# 메타데이터 필터링 — 의미 검색 + 정확한 카테고리 필터를 동시에 적용한다.
# `where={"doc_type": "department"}` 는 doc_type 메타데이터가 'department' 인 문서만 검색 대상.
# 일반 SQL 의 WHERE 절과 비슷한 역할.
filtered = collection.query(
    query_texts=["의사 정보를 알려주세요"],
    n_results=5,
    where={"doc_type": "department"},
)
print("Filter: doc_type=department\n")
for doc, meta in zip(filtered["documents"][0], filtered["metadatas"][0]):
    print(f"  [{meta['section']}] {doc}")

## LlamaIndex + ChromaDB 통합

`ChromaVectorStore` 로 LlamaIndex 인덱스의 스토리지를 Chroma 컬렉션으로 지정합니다. 이제 인덱스가 디스크에 영속화됩니다.

In [ ]:
# LlamaIndex 에 Chroma 를 "벡터 저장소" 로 끼워 넣어, 인덱스 자체가 디스크에 영속되도록 만든다.
# 04번 노트북의 인메모리 인덱스와 달리, 노트북을 재시작해도 인덱스를 다시 만들 필요가 없어집니다.
from llama_index.core import VectorStoreIndex, StorageContext, Document, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI as LIOpenAI
# `as LIOpenAI` 는 임포트 별칭. 위에서 이미 `from openai import OpenAI` 로 다른 클래스를 들여왔기 때문에
# 이름 충돌을 피하려고 LlamaIndex 의 OpenAI 래퍼는 LIOpenAI 라는 다른 이름으로 받습니다.

Settings.llm = LIOpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# LlamaIndex 전용 컬렉션을 별도로 만든다 — 위의 `hospital_docs` 와 분리하면 데이터 형식 충돌이 없다.
chroma_collection = chroma_client.get_or_create_collection("hospital_llamaindex")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
# StorageContext 는 "이 인덱스가 어디에 저장되는지"를 한데 묶어 알려 주는 객체.
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 위에서 만들어 둔 documents/metadatas 를 LlamaIndex Document 형식으로 다시 포장해 적재.
li_docs = [Document(text=d, metadata=m) for d, m in zip(documents, metadatas)]
index = VectorStoreIndex.from_documents(
    li_docs,
    storage_context=storage_context,
    show_progress=True,
)
print("LlamaIndex + ChromaDB index ready.")

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)
resp = query_engine.query("응급실 이용 가능한 시간은?")
print("Answer:", resp.response)
print("\nSources:")
for node in resp.source_nodes:
    print(f"  - score={node.score:.3f}: {node.text[:60]}...")

## 영속성 확인

동일한 경로로 새로운 `PersistentClient` 를 만들어 **문서 수가 그대로 남아있는지** 검증합니다. Colab 세션이 끊겨도 `./chroma_db` 폴더가 남아 있다면 데이터는 보존됩니다.

In [ ]:
# 같은 경로로 새 클라이언트를 다시 열어서 데이터가 디스크에 잘 보관됐는지 확인.
# 노트북 재시작 후에도 이 셀을 처음부터 돌리면 카운트가 0이 아니라 8로 나옵니다 — 영속화 성공의 증거.
chroma_client2 = chromadb.PersistentClient(path="./chroma_db")
collection2 = chroma_client2.get_collection("hospital_docs")
print(f"Reconnect check — hospital_docs count: {collection2.count()}")

## 임베딩 모델 비교

| 모델 | 차원 | 비용 | 특징 |
|---|---|---|---|
| `text-embedding-3-small` (OpenAI) | 1,536 | 저렴 | 기본값, 빠름 |
| `text-embedding-3-large` (OpenAI) | 3,072 | 보통 | 더 높은 정확도 |
| `BAAI/bge-small-ko` (HuggingFace) | 384 | 무료 | 한국어 특화, 경량 |
| `sentence-transformers/paraphrase-multilingual` (HF) | 768 | 무료 | 다국어 |

비용과 정확도의 트레이드오프이며, 프로젝트 초기에는 `3-small` 로 시작해 필요 시 업그레이드하는 전략이 무난합니다.

## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 의료 용어 유사도 실험

의료 도메인에서 유사한 표현들의 임베딩 유사도를 실험해보세요.

테스트할 용어 6개:

- "고혈압", "혈압이 높다"
- "두통", "머리가 아프다"
- "당뇨병", "혈당이 높다"

_힌트: 위 6개 용어를 리스트로 만들고 각각 `get_embedding(t)`로 임베딩한 뒤, 이중 for 루프(`i, j with i<j`)로 모든 쌍에 대해 `cosine_similarity()`를 출력하세요._

**관찰 포인트:**

- "고혈압" vs "혈압이 높다" → 높은 유사도 (같은 의미, 다른 표현)
- "고혈압" vs "두통" → 중간 유사도 (고혈압의 증상으로 두통이 올 수 있음)
- "고혈압" vs "당뇨병" → 중간 유사도 (둘 다 만성질환)
- 이런 의미적 유사성을 AI가 자동으로 파악하는 것이 임베딩의 핵심입니다


In [ ]:
# ============================================================
# 실습 과제 — 의료 용어 임베딩 유사도
# ============================================================

# 실습 1: 의료 용어 유사도 실험
# TODO: ["고혈압", "혈압이 높다", "두통", "머리가 아프다", "당뇨병", "혈당이 높다"]
#       6개 용어를 get_embedding()으로 임베딩하고, 이중 for 루프(i<j)로 모든 쌍의
#       cosine_similarity()를 출력하세요.
# 여기에 구현하세요.


## 다음 노트북에서는…

지금까지는 "문서" 기반 RAG 였습니다. 다음 **`06_text_to_sql.ipynb`** 에서는 LlamaIndex 의 `SQLDatabase` 래퍼와 `NLSQLTableQueryEngine` 으로 **DB 에 자연어 질의** 를 실행해 봅니다. 3H 에서 다듬은 `COMMENT ON` 이 여기서 진가를 발휘합니다.